<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum02/BD_KelasA_P02_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BD_KelasA_P02_2411531001_Habibi Putra Rizqullah

**Praktikum 2 - Pengumpulan dan Pra-pemrosesan Data (Data Acquisition & Preprocessing)**

Mata Kuliah Praktikum Big Data - S1 Informatika Semester V

## K-1. Import Library dan Inisialisasi

In [38]:
!pip install faker -q

In [39]:
import numpy as np
import pandas as pd
from faker import Faker
import random

## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [40]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [41]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-07-11,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-07-07,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-08-23,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,25/07/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,29/06/2026,NaN,1.0


## K-3. Deteksi dan Penanganan Missing Value

In [42]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [43]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah penanganan missing value:", len(df))

Jumlah baris setelah penanganan missing value: 495


## K-4. Deteksi dan Penanganan Duplicate

In [44]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## K-5. Koreksi Tipe Data dan Standardisasi Format

### a. Standardisasi teks kategorikal (category, payment_method, shipping_city)

In [45]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())

<StringArray>
['Buku', 'Elektronik', 'Fashion', 'Rumah Tangga', 'Kesehatan', 'Olahraga']
Length: 6, dtype: string
<StringArray>
['Kartu Kredit', 'E-Wallet', 'COD', 'Transfer Bank']
Length: 4, dtype: string


### b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)

In [46]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)
df["price"].head()

,price
0,50000.0
1,500000.0
2,25000.0
3,250000.0
5,250000.0


### c. Standardisasi format tanggal ke YYYY-MM-DD

In [47]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df["transaction_date"].head()

,transaction_date
0,2026-07-11
1,2026-07-07
2,2026-08-23
3,2026-07-25
5,2026-09-13


### d. Finalisasi tipe data

In [48]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)
df.dtypes

,0
transaction_id,object
customer_name,object
product_name,object
category,string[python]
price,float64
quantity,int64
payment_method,string[python]
transaction_date,object
shipping_city,string[python]
rating,float64


## K-6. Ekspor Dataset Bersih

In [49]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


## Latihan 1 Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris `transaksi_mentah.csv` dan `transaksi_bersih.csv` dengan hasil SEED = 42.

In [58]:
def jalankan_pipeline(seed):
    np.random.seed(seed)
    random.seed(seed)
    fk = Faker("id_ID")
    Faker.seed(seed)

    rows_ = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fk.name()
        produk = fk.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)
        harga_variants = [str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."), f"{harga_dasar}.0", f" {harga_dasar} "]
        harga = random.choice(harga_variants)
        tgl = fk.date_between(start_date="-90d", end_date="today")
        tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
        tanggal = random.choice(tgl_variants)
        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()
        if random.random() < 0.2:
            kategori = kategori.upper() + "  "
        kota = fk.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])
        rows_.append({"transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
                       "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
                       "transaction_date": tanggal, "shipping_city": kota, "rating": rating})

    d = pd.DataFrame(rows_)
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        idx = d.sample(frac=frac, random_state=seed).index
        d.loc[idx, col] = np.nan
    dup = d.sample(n=15, random_state=seed)
    d = pd.concat([d, dup], ignore_index=True)
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_mentah = len(d)

    d = d.dropna(subset=["customer_name", "payment_method"])
    d["shipping_city"] = d["shipping_city"].fillna("Tidak Diketahui")
    d = d.drop_duplicates()
    n_bersih = len(d)
    return n_mentah, n_bersih

mentah_42, bersih_42 = 515, len(df)
mentah_7, bersih_7 = jalankan_pipeline(7)

print(f"SEED=42 -> transaksi_mentah: {mentah_42} baris, transaksi_bersih: {bersih_42} baris")
print(f"SEED=7  -> transaksi_mentah: {mentah_7} baris, transaksi_bersih: {bersih_7} baris")

SEED=42 -> transaksi_mentah: 515 baris, transaksi_bersih: 490 baris
SEED=7  -> transaksi_mentah: 515 baris, transaksi_bersih: 490 baris


## Latihan 2 Tambahkan kolom `is_valid_price` bernilai True jika price > 0.

In [54]:
df["is_valid_price"] = df["price"] > 0
print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0




## Latihan 3 Hitung jumlah transaksi per category menggunakan value_counts().

In [57]:
print(df["category"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
